In [6]:
# ==========================================================
# CELL 0: MOUNT GOOGLE DRIVE + CÀI ĐẶT OLLAMA + TẢI LLAMA 3.2 3B
# (Mount Drive để đọc file "Gen dataset.json" của bạn trên đó.
#  Ollama thì không cần Drive, chạy thẳng trên Colab.)
# ==========================================================

# 0. Mount Google Drive (file Gen dataset.json của bạn đang nằm trong
#    MyDrive/Project DAP/Gen dataset.json)
from google.colab import drive
drive.mount('/content/drive')

!pip install xgboost lightgbm sentence-transformers scikit-learn

# 1. Cài công cụ giải nén zstd (Ollama cần để cài đặt đúng)
!apt-get install -y zstd

# 2. Cài Ollama vào máy ảo Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Chạy Ollama server dưới nền (background), lắng nghe ở localhost:11434
import subprocess
import time
subprocess.Popen(["ollama", "serve"])

# Chờ vài giây để service khởi động xong
time.sleep(5)

# 4. Tải model llama3.2:3b về Colab (đây là server Colab riêng, không liên quan
#    tới Ollama trên máy Windows của bạn — nó sẽ tự pull lại model trên cloud)
!ollama pull llama3.2:3b

# 5. "Làm nóng" model: gọi thử 1 lần để Ollama load hẳn model vào RAM/VRAM ngay bây giờ.
#    Nếu không làm bước này, request ĐẦU TIÊN ở Cell 1 sẽ phải tự load model
#    (mất khoảng 30s-1 phút) và dễ bị timeout. Lần gọi này có thể mất chút thời gian,
#    đó là bình thường — cứ để nó chạy xong.
import requests
print("Đang load model vào memory (làm nóng), có thể mất 30s-1 phút...")
try:
    warmup = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3.2:3b", "prompt": "Xin chào", "stream": False},
        timeout=180  # cho phép chờ tới 3 phút cho lần load đầu tiên
    )
    warmup.raise_for_status()
    print("✅ Model đã load xong vào memory, sẵn sàng trả lời nhanh ở các lần gọi sau.")
except Exception as e:
    print(f"⚠️ Làm nóng model thất bại: {e}")
    print("   Không sao, Cell 1 vẫn có thể tự thử lại, nhưng nếu lỗi lặp lại,")
    print("   chạy lại cell này (Cell 0) một lần nữa.")

print("\n✅ Ollama đã sẵn sàng, model llama3.2:3b đã tải và load xong trên Colab.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Đang load model vào memory (làm nóng), có thể mất 30s-1 phút...
✅ Model đã load xong vào memory, sẵn sàng trả lời nhanh ở các lần gọi sau.

✅ Ollama đã sẵn sàng, model llama3.2:3b đã tải và load xong trên

In [ ]:
# ==========================================================
# CELL NÂNG CẤP: MACHINE LEARNING (STACKING) + LLM (LLAMA 3.2)
# ==========================================================

import json
import torch
import random
import requests
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Các thư viện Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

# ==========================================================
# CẤU HÌNH THAM SỐ
# ==========================================================
PROB_THRESHOLD = 0.15  # Ngưỡng xác suất để chặn câu hỏi Lạc đề (Thay cho Cosine Threshold)
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL_NAME = "llama3.2:3b"

# ==========================================================
# 1. ĐỌC VÀ CHIA DATASET
# ==========================================================
print("--- 1. Đang tải và phân tách dữ liệu ---")
with open('/content/drive/MyDrive/Project DAP/Gen_dataset_v6.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

records = []
for intent in data['intents']:
    tag = intent['tag']
    for pattern in intent['patterns']:
        records.append({'pattern': pattern, 'intent': tag})

df = pd.DataFrame(records)
X = df['pattern']
y = df['intent']

# BẮT BUỘC: Mã hóa nhãn (Intent) thành số (0, 1, 2...) cho XGBoost & LightGBM
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=123, stratify=y_encoded
)

# ==========================================================
# 2. VECTOR HÓA DỮ LIỆU (BGE-M3)
# ==========================================================
print("\n--- 2. Tải mô hình Embedding (BGE-M3) ---")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_model = SentenceTransformer('BAAI/bge-m3', device=device)

print("Đang chuyển đổi dữ liệu chữ sang Vector...")
X_train_vectors = embed_model.encode(X_train.tolist(), show_progress_bar=True)
X_test_vectors = embed_model.encode(X_test.tolist(), show_progress_bar=True)

# ==========================================================
# 3. KIỂM TRA OLLAMA
# ==========================================================
print(f"\n--- 3. Kiểm tra Ollama ({OLLAMA_MODEL_NAME}) ---")
try:
    requests.post(OLLAMA_URL, json={"model": OLLAMA_MODEL_NAME, "prompt": "Hi", "stream": False}, timeout=10)
    print("-> Kết nối Ollama thành công!")
except Exception as e:
    print(f"❌ Lỗi kết nối Ollama: {e}")

# ==========================================================
# 4. HUẤN LUYỆN MACHINE LEARNING (GRIDSEARCH & STACKING)
# ==========================================================
print("\n--- 4. BẮT ĐẦU HUẤN LUYỆN VÀ TUNING MÔ HÌNH ML ---")

# Không gian tham số (Tối giản để chạy nhanh trên Colab)
param_grids = {
    "SVM (Linear)": {
        'model': SVC(probability=True, random_state=42),
        'params': {'C': [10.0], 'kernel': ['linear']}
    },
    "Random Forest": {
        'model': RandomForestClassifier(random_state=42),
        'params': {'n_estimators': [100, 200], 'max_depth': [None, 10]}
    },
    "LightGBM": {
        'model': LGBMClassifier(random_state=42, verbose=-1),
        'params': {'n_estimators': [100], 'learning_rate': [0.05, 0.1]}
    },
    "XGBoost": {
        'model': XGBClassifier(random_state=42, eval_metric='mlogloss'),
        'params': {'n_estimators': [100], 'learning_rate': [0.1], 'max_depth': [3]}
    }
}

best_models = {}
for name, config in param_grids.items():
    print(f" Đang Tuning {name}...")
    grid = GridSearchCV(config['model'], config['params'], cv=3, scoring='f1_weighted', n_jobs=-1, verbose = 3)
    grid.fit(X_train_vectors, y_train)
    best_models[name] = grid.best_estimator_

print("\n Đang huấn luyện Stacking Classifier (Kết hợp 4 mô hình)...")
estimators = [(name, model) for name, model in best_models.items()]
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3, n_jobs=-1, verbose = 2
)
stacking_clf.fit(X_train_vectors, y_train)
best_models["Stacking Classifier"] = stacking_clf
print("-> Hoàn tất Huấn luyện!")

# ==========================================================
# 5. ĐÁNH GIÁ KẾT QUẢ (4 METRICS)
# ==========================================================
print("\n" + "="*70)
print(f"{'MÔ HÌNH':<22} | {'ACCURACY':<8} | {'F1 SCORE':<8} | {'RECALL':<8} | {'PRECISION':<8}")
print("="*70)

metrics_report = {}
for name, model in best_models.items():
    y_pred = model.predict(X_test_vectors)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)

    metrics_report[name] = f1
    print(f"{name:<22} | {acc:.4f}   | {f1:.4f}   | {rec:.4f}   | {prec:.4f}")
print("="*70)

# Chốt mô hình có F1 cao nhất để làm não bộ Chatbot
best_model_name = max(metrics_report, key=metrics_report.get)
final_ml_model = best_models[best_model_name]
print(f"🏆 CHỐT MÔ HÌNH CHO CHATBOT: {best_model_name}\n")

# ==========================================================
# 6. GIAO DIỆN CHAT TRỰC TIẾP TÍCH HỢP LLM
# ==========================================================
print("--- 6. GIAO DIỆN CHAT THỰC TẾ ---")

intent_to_responses = {intent['tag']: intent.get('responses', []) for intent in data['intents']}

input_text = widgets.Text(placeholder='Hãy thử đổi từ, dùng câu đồng nghĩa...', description='Bạn:')
button = widgets.Button(description='Gửi câu chat')
output = widgets.Output()

output.layout.max_height = '400px'
output.layout.overflow = 'auto'
output.layout.border = '1px solid #ccc'
output.layout.padding = '10px'

with output:
    print("Bot Stacking ML + LLM đã sẵn sàng!")
    print("-" * 60)

def generate_llm_response(user_query, retrieved_context):
    prompt = f'''You are a helpful and empathetic assistant. Answer the user's question based ONLY on the provided context. Keep it natural and concise (under 3 sentences).

Context: {retrieved_context}

Question: {user_query}

Answer:'''

    payload = {"model": OLLAMA_MODEL_NAME, "prompt": prompt, "stream": False, "options": {"temperature": 0.7}}
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        return response.json()["response"].strip()
    except Exception as e:
        return f"(Lỗi gọi LLM qua Ollama: {e})"

def on_submit(b):
    user_input = input_text.value.strip()
    if not user_input: return

    with output:
        # 1. Vector hóa câu người dùng
        input_vector = embed_model.encode([user_input], device=device)

        # 2. ML dự đoán XÁC SUẤT
        probabilities = final_ml_model.predict_proba(input_vector)[0]
        max_prob_index = np.argmax(probabilities)
        max_prob_score = probabilities[max_prob_index]

        # 3. KIỂM TRA THRESHOLD LẠC ĐỀ
        if max_prob_score < PROB_THRESHOLD:
            print(f"Bạn: {user_input}")
            print(f"Bot: Xin lỗi, câu hỏi này nằm ngoài phạm vi hiểu biết hiện tại của hệ thống.")
            print(f"   \033[91m[Hệ thống từ chối - Độ tự tin: {max_prob_score:.2f} < Ngưỡng an toàn: {PROB_THRESHOLD}]\033[0m")
            print("-" * 60)
        else:
            # Nếu an toàn, dịch từ số (0,1,2) sang chữ (Intent Tag)
            pred_intent = label_encoder.inverse_transform([max_prob_index])[0]

            # Lấy Context cho RAG
            context_responses = intent_to_responses.get(pred_intent, [""])
            context_text = " ".join(context_responses[:2])

            print("Bot đang suy nghĩ...")
            bot_llm_response = generate_llm_response(user_input, context_text)

            clear_output(wait=True)
            print(f"Bạn: {user_input}")
            print(f"Bot (LLM sinh câu): {bot_llm_response}")
            print(f"   \033[92m[Debug - ML gán Tag: {pred_intent} | Xác suất (Confidence): {max_prob_score:.2f}]\033[0m")
            print("-" * 60)

    input_text.value = ""

button.on_click(on_submit)
input_text.on_submit(on_submit)

display(widgets.VBox([output, widgets.HBox([input_text, button])]))

--- 1. Đang tải và phân tách dữ liệu ---

--- 2. Tải mô hình Embedding (BGE-M3) ---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Đang chuyển đổi dữ liệu chữ sang Vector...


Batches:   0%|          | 0/137 [00:00<?, ?it/s]

Batches:   0%|          | 0/35 [00:00<?, ?it/s]


--- 3. Kiểm tra Ollama (llama3.2:3b) ---
-> Kết nối Ollama thành công!

--- 4. BẮT ĐẦU HUẤN LUYỆN VÀ TUNING MÔ HÌNH ML ---
 Đang Tuning SVM (Linear)...
 Đang Tuning Random Forest...
 Đang Tuning LightGBM...
 Đang Tuning XGBoost...

 Đang huấn luyện Stacking Classifier (Kết hợp 4 mô hình)...
-> Hoàn tất Huấn luyện!

MÔ HÌNH                | ACCURACY | F1 SCORE | RECALL   | PRECISION
SVM (Linear)           | 0.9239   | 0.9242   | 0.9239   | 0.9312
Random Forest          | 0.8927   | 0.8889   | 0.8927   | 0.8967
LightGBM               | 0.8578   | 0.8570   | 0.8578   | 0.8678
XGBoost                | 0.8716   | 0.8733   | 0.8716   | 0.8858
Stacking Classifier    | 0.8917   | 0.8949   | 0.8917   | 0.9104
🏆 CHỐT MÔ HÌNH CHO CHATBOT: SVM (Linear)

--- 6. GIAO DIỆN CHAT THỰC TẾ ---


In [5]:
# ==========================================================
# CELL 3: PHÂN TÍCH LỖI CHI TIẾT (Cho Workflow Machine Learning)
# ==========================================================
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.metrics import classification_report

print("--- ĐANG PHÂN TÍCH LỖI TRÊN MÔ HÌNH CHỐT: {} ---".format(best_model_name))

# 1. Lấy dự đoán và Xác suất từ mô hình ML tốt nhất
y_pred_probs = final_ml_model.predict_proba(X_test_vectors)
max_probs = np.max(y_pred_probs, axis=1) # Lấy xác suất cao nhất của từng câu
pred_indices = np.argmax(y_pred_probs, axis=1) # Lấy vị trí (nhãn số) của xác suất đó

# 2. Dịch ngược nhãn số về dạng chữ (Text) ban đầu
y_test_true_text = label_encoder.inverse_transform(y_test)
y_pred_text_raw = label_encoder.inverse_transform(pred_indices)

# 3. Áp dụng Threshold Lạc đề (Fallback)
y_pred_final = []
for i in range(len(y_test_true_text)):
    if max_probs[i] < PROB_THRESHOLD:
        y_pred_final.append("unknown_fallback")
    else:
        y_pred_final.append(y_pred_text_raw[i])

# ==========================================
# BÁO CÁO 1: CÁC CÂU BỊ LOẠI OAN
# ==========================================
X_test_list = X_test.tolist() # Ép kiểu về List để gọi index [i] không bị lỗi

wrongly_rejected = [
    (X_test_list[i], y_test_true_text[i], max_probs[i])
    for i in range(len(y_test_true_text))
    if y_pred_final[i] == "unknown_fallback"
]
print(f"\n[1] Số câu bị THRESHOLD loại oan: {len(wrongly_rejected)} / {len(y_test_true_text)} "
      f"({len(wrongly_rejected)/len(y_test_true_text)*100:.1f}%)")
if wrongly_rejected:
    print("Ví dụ 5 câu bị loại oan (Câu gốc -> Intent thật | Độ tự tin):")
    for ex in sorted(wrongly_rejected, key=lambda x: -x[2])[:5]:
        print(f"  '{ex[0]}' -> Thật là [{ex[1]}], Score={ex[2]:.3f}")

# ==========================================
# BÁO CÁO 2: NHẦM LẪN GIỮA CÁC INTENT
# ==========================================
confusions = [
    (y_test_true_text[i], y_pred_final[i])
    for i in range(len(y_test_true_text))
    if y_pred_final[i] != y_test_true_text[i] and y_pred_final[i] != "unknown_fallback"
]
print(f"\n[2] Số câu model nhầm sang intent KHÁC (không tính bị threshold loại): "
      f"{len(confusions)} / {len(y_test_true_text)}")
confusion_counter = Counter(confusions)
print("Top 30 cặp (Intent thật -> Intent dự đoán nhầm) xuất hiện nhiều nhất:")
for (true_label, pred_label), count in confusion_counter.most_common(30):
    print(f"  {true_label}  ->  {pred_label}   ({count} câu)")

# ==========================================
# BÁO CÁO 3: TOP 30 INTENT YẾU NHẤT
# ==========================================
# Điều chỉnh nhãn test thật để tính toán công bằng (Nếu đoán unknown thì nhãn thật cũng phải xem là unknown)
y_test_adjusted_text = [
    y_test_true_text[i] if y_pred_final[i] != "unknown_fallback" else "unknown_fallback"
    for i in range(len(y_test_true_text))
]

report = classification_report(y_test_adjusted_text, y_pred_final, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose().sort_values('f1-score')
print("\n[3] TOP 30 Intent có F1-score THẤP NHẤT (Cần xem lại dữ liệu):")
print(report_df.head(30)[['precision', 'recall', 'f1-score', 'support']])

--- ĐANG PHÂN TÍCH LỖI TRÊN MÔ HÌNH CHỐT: SVM (Linear) ---

[1] Số câu bị THRESHOLD loại oan: 877 / 1090 (80.5%)
Ví dụ 5 câu bị loại oan (Câu gốc -> Intent thật | Độ tự tin):
  'I need to get my aging father some professional help, where can older adults find help for mental health concerns?' -> Thật là [faq_1833460], Score=0.600
  'medication affordability' -> Thật là [faq_1546812], Score=0.600
  'Is there any solid research proving its benefits, and what is the legal status and evidence of CBD oil?' -> Thật là [faq_6521784], Score=0.599
  'Can you describe the concept of mental health?' -> Thật là [what_is_mental_health], Score=0.599
  'I repeated my order at the restaurant' -> Thật là [control_repeat], Score=0.599

[2] Số câu model nhầm sang intent KHÁC (không tính bị threshold loại): 0 / 1090
Top 30 cặp (Intent thật -> Intent dự đoán nhầm) xuất hiện nhiều nhất:

[3] TOP 30 Intent có F1-score THẤP NHẤT (Cần xem lại dữ liệu):
                      precision  recall  f1-score  support